In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
import logging

LOGGER = logging.getLogger(__name__)

logging.basicConfig(level=logging.CRITICAL, format="%(name)s %(asctime)s %(message)s")
LOGGER.setLevel(logging.INFO)

In [37]:
from utils.database_utils import generate_database_and_retriever, populate_database
from scipy.spatial.distance import cosine

In [38]:
from utils.node_standarization import translate_nodes

In [39]:
data_base = "./localdb"
retriever = generate_database_and_retriever(main_folder=data_base)

all_keys = list(retriever.docstore.yield_keys())
all_documents = retriever.docstore.mget(all_keys)
docutments_dic = {all_keys[i]: all_documents[i] for i in range(len(all_keys))}

backoff 2026-03-18 07:25:49,450 Backing off send_request(...) for 0.3s (requests.exceptions.ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))


In [40]:
import pickle

with open("graph_raw_data.pkl", "rb") as f:
    graph_elements = pickle.load(f)

In [ ]:
graph_elements_translated = translate_nodes(graph_elements)

In [25]:
from utils.node_standarization import lemmatize_nodes_and_relationships

graph_elements_lematized = lemmatize_nodes_and_relationships(graph_elements_translated)

In [27]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage
from typing import List
from collections import Counter


class EmbeddingNode:
    model_name: str

    def __init__(self, model_name):
        self.model = OllamaEmbeddings(model=model_name)

    def get_embedding(self, text):
        return self.model.embed_query(text)


class DescriptionNode:
    def __init__(self, model_name):
        self.model = OllamaLLM(model=model_name)
        self.system_instruction = """
            You are an expert Graph Data Analyst. Your task is to extract a simple description that summarizes of all the provided descriptions. 
            ### Output Format
            The output should be a single string that summarizes all the provided descriptions.
        """
        self.prompt = """
            Descriptions: {descriptions}
        """

    def get_description(self, text):
        formatted_prompt = self.prompt.format(descriptions=text)
        messages = [
            SystemMessage(content=self.system_instruction),
            HumanMessage(content=formatted_prompt),
        ]
        summary = self.model.invoke(messages)
        return summary


class Node:
    embedding: List[int]
    name: str
    type: str
    description: str
    model: EmbeddingNode
    node_id: str = None

    def __init__(self, name, type, description, model, node_id):
        self.name = name
        self.type = type
        self.description = description
        self.model = model
        self.node_id = node_id

    def get_embedding(self):
        self.embedding = self.model.get_embedding(self.name)


class RepresentativeNode:
    group_id: str
    mapping_nodes: List[str]
    description: str = None
    name: str = None
    type: str = None

    def __init__(self, group_id, all_nodes, model):
        self.group_id = group_id
        group_nodes = [node for node in all_nodes if node.group_id == group_id]
        all_descriptions = [node.description for node in group_nodes]
        unique_descriptions = set(all_descriptions)
        if len(unique_descriptions) == 1:
            self.description = list(unique_descriptions)[0]

        else:
            self.description = model.get_description(
                "\n".join(list(unique_descriptions))
            )

        type = Counter(node.type for node in group_nodes)
        self.type = type.most_common(1)[0][0]

        name = Counter(node.name for node in group_nodes)
        self.name = name.most_common(1)[0][0]

        self.mapping_nodes = [
            node.node_id for node in group_nodes
        ]  ## This will need to be the node id


In [28]:
import tqdm

In [29]:
import enum

from pyvis import node


all_nodes = []
embedding_model = EmbeddingNode(model_name="embeddinggemma:latest")
for document_id, relations in tqdm.tqdm(
    graph_elements_lematized.items(), total=len(graph_elements_lematized)
):
    for index_relation, relationship in enumerate(relations):
        head_id = document_id + "_" + "head" + "_" + str(index_relation)
        tail_id = document_id + "_" + "tail" + "_" + str(index_relation)
        node_head = Node(
            name=relationship.head,
            type=relationship.head_type,
            description=relationship.head_description,
            model=embedding_model,
            node_id=head_id,
        )
        relationship.head_id = head_id
        node_head.get_embedding()
        all_nodes.append(node_head)

        node_tail = Node(
            name=relationship.tail,
            type=relationship.tail_type,
            description=relationship.tail_description,
            model=embedding_model,
            node_id=tail_id,
        )
        relationship.tail_id = tail_id
        node_tail.get_embedding()
        all_nodes.append(node_tail)


100%|██████████| 9/9 [00:10<00:00,  1.19s/it]


In [30]:
class UnionFind:
    def __init__(self, nodes):
        self.parent = {node.id: node.id for node in nodes}

    def find(self, i):
        if self.parent[i] == i:
            return i
        self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            self.parent[root_i] = root_j


# 1. Ensure initial state is None
for node in all_nodes:
    node.group_id = None

for index, node in enumerate(all_nodes):
    node.id = "node_" + str(index)

uf = UnionFind(all_nodes)
threshold = 0.05

# 2. Perform the comparisons
for i in range(len(all_nodes)):
    for j in range(i + 1, len(all_nodes)):
        node1, node2 = all_nodes[i], all_nodes[j]
        if cosine(node1.embedding, node2.embedding) < threshold:
            uf.union(node1.id, node2.id)

# 3. Final Assignment
# This converts the internal 'parent' pointers into a final, shared ID
for node in all_nodes:
    node.group_id = uf.find(node.id)

nodes_by_group = {}
for node in all_nodes:
    group_id = node.group_id
    if group_id not in nodes_by_group:
        nodes_by_group[group_id] = []
    nodes_by_group[group_id].append(node)


In [31]:
model_for_descriptions = DescriptionNode(model_name="gemma3:12b")
nodes_representatives = []
for group_id, nodes in tqdm.tqdm(nodes_by_group.items(), total=len(nodes_by_group)):
    nodes_representatives.append(
        RepresentativeNode(group_id, nodes, model_for_descriptions)
    )

100%|██████████| 60/60 [00:27<00:00,  2.20it/s]


# Reconstruc the clean graph

In [32]:
for node_representative in nodes_representatives:
    for document_id, relations in graph_elements_lematized.items():
        for index_relation, relationship in enumerate(relations):
            if relationship.head_id in node_representative.mapping_nodes:
                relationship.head = node_representative.name
                relationship.head_type = node_representative.type
                relationship.head_description = node_representative.description
            if relationship.tail_id in node_representative.mapping_nodes:
                relationship.tail = node_representative.name
                relationship.tail_type = node_representative.type
                relationship.tail_description = node_representative.description

In [33]:
name_to_representative = {}

for node in all_nodes:
    root_id = uf.find(node.id)
    # Find the node that actually owns this root_id to get its 'canonical' name
    # Or just use the root_id itself as the new name
    name_to_representative[node.name] = root_id

In [34]:
import pickle

with open("graph_clean_data.pkl", "wb") as f:
    pickle.dump(graph_elements_lematized, f)